# Initial Exploratory Data Analysis (Diagnostic EDA)

---

## 📌 Overview
Initial Exploratory Data Analysis serves as a diagnostic health check performed immediately after data ingestion and prior to any data cleaning or transformation. 

The primary objective of this phase is to thoroughly inspect the "raw state" of our datasets in Python. By understanding structural flaws, missingness, datatypes, and initial statistical distributions, we can design a precise, informed data cleaning strategy.



In [1]:
import os
import urllib.parse
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. Connection (Uses project root automatically)
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME")

ENCODED_PASSWORD = urllib.parse.quote_plus(DB_PASSWORD)
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{ENCODED_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("✅ Connected to MySQL successfully!")


def inspect_table(table_name, pk_col=None):
    """
    Reads a table from MySQL and prints a complete structural, statistical,
    and categorical profile (Read-Only).
    """
    df = pd.read_sql(f"SELECT * FROM {table_name}", con=engine)

    print("=" * 80)
    print(f"📋 READ-ONLY INITIAL PROFILE: '{table_name.upper()}'")
    print("=" * 80)

    # 1. Shape & Memory
    memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f"• Shape: {df.shape[0]:,} rows | {df.shape[1]} columns")
    print(f"• Memory Usage: {memory_mb:.2f} MB")
    print(f"• Total Duplicate Rows: {df.duplicated().sum():,}")

    # 2. Primary Key Uniqueness Check (Point 6)
    if pk_col:
        if pk_col in df.columns:
            is_unique = df[pk_col].is_unique
            dup_pk_count = df[pk_col].duplicated().sum()
            status = "✅ YES (Unique)" if is_unique else "❌ NO (Duplicates Found!)"
            print(f"• Primary Key ('{pk_col}') Status: {status} | Duplicates: {dup_pk_count}")
        else:
            print(f"⚠️ Specified Primary Key '{pk_col}' not found in columns.")

    print("-" * 80)

    # 3. Schema & Nulls Summary
    summary_df = pd.DataFrame(
        {
            "Data Type": df.dtypes,
            "Null Count": df.isnull().sum(),
            "Null %": (df.isnull().sum() / len(df) * 100).round(2),
            "Unique Values": df.nunique(),
        }
    )
    print("\n🔍 Column Summary:")
    print(summary_df)
    print("-" * 80)

    # 4. Date / Timestamp Range Detection (Point 4)
    # Detects string columns containing 'date' or 'timestamp' and converts temporarily to check ranges
    date_cols = [
        col for col in df.columns if any(k in col.lower() for k in ["date", "timestamp", "time"])
    ]
    if date_cols:
        print("\n📅 Date & Timestamp Ranges:")
        for col in date_cols:
            temp_dates = pd.to_datetime(df[col], errors="coerce")
            if temp_dates.notnull().any():
                print(f"  * {col}: Min = {temp_dates.min()} | Max = {temp_dates.max()}")
        print("-" * 80)

    # 5. Top Categorical Values Distribution (Point 5)
    cat_cols = df.select_dtypes(include=["object"]).columns
    # Exclude IDs/PKs from top-5 value prints
    value_cols = [col for col in cat_cols if not col.endswith("_id") and col != pk_col]

    if value_cols:
        print("\n📊 Top 5 Value Counts (Categorical Columns):")
        for col in value_cols[:4]:  # Limits output to first 4 relevant columns to avoid clutter
            print(f"\n--- Column: {col} ---")
            print(df[col].value_counts().head(5))
        print("-" * 80)

    # 6. Numerical Summary
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        print("\n📈 Numerical Summary:")
        display(df[num_cols].describe().T)

    print("=" * 80 + "\n\n")
    return df

✅ Connected to MySQL successfully!


In [2]:
# 1. Core Fact Table
orders_df = inspect_table("olist_orders", pk_col="order_id")

# 2. Order Line Items
order_items_df = inspect_table("olist_order_items")

# 3. Customers
customers_df = inspect_table("olist_customers", pk_col="customer_id")

# 4. Products
products_df = inspect_table("olist_products", pk_col="product_id")

# 5. Payments
payments_df = inspect_table("olist_order_payments")

# 6. Reviews
reviews_df = inspect_table("olist_order_reviews", pk_col="review_id")

# 7. Sellers
sellers_df = inspect_table("olist_sellers", pk_col="seller_id")

# 8. Category Translation
translation_df = inspect_table("product_category_translation")

# 9. Geolocation
geolocation_df = inspect_table("olist_geolocation")

📋 READ-ONLY INITIAL PROFILE: 'OLIST_ORDERS'
• Shape: 99,441 rows | 8 columns
• Memory Usage: 24.66 MB
• Total Duplicate Rows: 0
• Primary Key ('order_id') Status: ✅ YES (Unique) | Duplicates: 0
--------------------------------------------------------------------------------

🔍 Column Summary:
                                    Data Type  Null Count  Null %  \
order_id                               object           0    0.00   
customer_id                            object           0    0.00   
order_status                           object           0    0.00   
order_purchase_timestamp       datetime64[ns]           0    0.00   
order_approved_at              datetime64[ns]         160    0.16   
order_delivered_carrier_date   datetime64[ns]        1783    1.79   
order_delivered_customer_date  datetime64[ns]        2965    2.98   
order_estimated_delivery_date  datetime64[ns]           0    0.00   

                               Unique Values  
order_id                             

,count,mean,std,min,25%,50%,75%,max
order_item_id,112650.0,1.197834,0.705124,1.00,1.00,1.00,1.00,21.00
price,112650.0,120.653739,183.633928,0.85,39.90,74.99,134.90,6735.00
freight_value,112650.0,19.990320,15.806405,0.00,13.08,16.26,21.15,409.68




📋 READ-ONLY INITIAL PROFILE: 'OLIST_CUSTOMERS'
• Shape: 99,441 rows | 5 columns
• Memory Usage: 26.59 MB
• Total Duplicate Rows: 0
• Primary Key ('customer_id') Status: ✅ YES (Unique) | Duplicates: 0
--------------------------------------------------------------------------------

🔍 Column Summary:
                         Data Type  Null Count  Null %  Unique Values
customer_id                 object           0     0.0          99441
customer_unique_id          object           0     0.0          96096
customer_zip_code_prefix     int64           0     0.0          14994
customer_city               object           0     0.0           4119
customer_state              object           0     0.0             27
--------------------------------------------------------------------------------

📊 Top 5 Value Counts (Categorical Columns):

--- Column: customer_city ---
customer_city
sao paulo         15540
rio de janeiro     6882
belo horizonte     2773
brasilia           2131
curitiba   

,count,mean,std,min,25%,50%,75%,max
customer_zip_code_prefix,99441.0,35137.474583,29797.938996,1003.0,11347.0,24416.0,58900.0,99990.0




📋 READ-ONLY INITIAL PROFILE: 'OLIST_PRODUCTS'
• Shape: 32,951 rows | 9 columns
• Memory Usage: 6.29 MB
• Total Duplicate Rows: 0
• Primary Key ('product_id') Status: ✅ YES (Unique) | Duplicates: 0
--------------------------------------------------------------------------------

🔍 Column Summary:
                           Data Type  Null Count  Null %  Unique Values
product_id                    object           0    0.00          32951
product_category_name         object         610    1.85             73
product_name_length          float64         610    1.85             66
product_description_length   float64         610    1.85           2960
product_photos_qty           float64         610    1.85             19
product_weight_g             float64           2    0.01           2204
product_length_cm            float64           2    0.01             99
product_height_cm            float64           2    0.01            102
product_width_cm             float64           2    0

...

,count,mean,std,min,25%,50%,75%,max
review_score,99224.0,4.086421,1.347579,1.0,4.0,5.0,5.0,5.0




📋 READ-ONLY INITIAL PROFILE: 'OLIST_SELLERS'
• Shape: 3,095 rows | 4 columns
• Memory Usage: 0.59 MB
• Total Duplicate Rows: 0
• Primary Key ('seller_id') Status: ✅ YES (Unique) | Duplicates: 0
--------------------------------------------------------------------------------

🔍 Column Summary:
                       Data Type  Null Count  Null %  Unique Values
seller_id                 object           0     0.0           3095
seller_zip_code_prefix     int64           0     0.0           2246
seller_city               object           0     0.0            611
seller_state              object           0     0.0             23
--------------------------------------------------------------------------------

📊 Top 5 Value Counts (Categorical Columns):

--- Column: seller_city ---
seller_city
sao paulo         694
curitiba          127
rio de janeiro     96
belo horizonte     68
ribeirao preto     52
Name: count, dtype: int64

--- Column: seller_state ---
seller_state
SP    1849
PR     

,count,mean,std,min,25%,50%,75%,max
seller_zip_code_prefix,3095.0,32291.059451,32713.45383,1001.0,7093.5,14940.0,64552.5,99730.0




📋 READ-ONLY INITIAL PROFILE: 'PRODUCT_CATEGORY_TRANSLATION'
• Shape: 71 rows | 2 columns
• Memory Usage: 0.01 MB
• Total Duplicate Rows: 0
--------------------------------------------------------------------------------

🔍 Column Summary:
                              Data Type  Null Count  Null %  Unique Values
product_category_name            object           0     0.0             71
product_category_name_english    object           0     0.0             71
--------------------------------------------------------------------------------

📊 Top 5 Value Counts (Categorical Columns):

--- Column: product_category_name ---
product_category_name
agro_industria_e_comercio    1
instrumentos_musicais        1
market_place                 1
malas_acessorios             1
livros_tecnicos              1
Name: count, dtype: int64

--- Column: product_category_name_english ---
product_category_name_english
agro_industry_and_commerce    1
musical_instruments           1
market_place             

,count,mean,std,min,25%,50%,75%,max
geolocation_zip_code_prefix,1000163.0,36574.166466,30549.335710,1001.000000,11075.000000,26530.000000,63504.000000,99990.000000
geolocation_lat,1000163.0,-21.176153,5.715866,-36.605374,-23.603546,-22.919377,-19.979620,45.065933
geolocation_lng,1000163.0,-46.390541,4.269748,-101.466766,-48.573172,-46.637879,-43.767709,121.105394


## 2. Relational & Foreign Key Integrity Checks

In [3]:
# Check for Orders without matching Customers
orphaned_orders = pd.read_sql("""
    SELECT COUNT(*) AS orphan_count 
    FROM olist_orders o 
    LEFT JOIN olist_customers c ON o.customer_id = c.customer_id 
    WHERE c.customer_id IS NULL
""", con=engine)

print(f"Orphaned Orders (No Customer): {orphaned_orders.iloc[0]['orphan_count']}")

Orphaned Orders (No Customer): 0


In [4]:
# 1. Order Items without Orders
items_no_orders = pd.read_sql("""
    SELECT COUNT(*) AS count 
    FROM olist_order_items i 
    LEFT JOIN olist_orders o ON i.order_id = o.order_id 
    WHERE o.order_id IS NULL
""", con=engine).iloc[0]['count']

# 2. Payments without Orders
payments_no_orders = pd.read_sql("""
    SELECT COUNT(*) AS count 
    FROM olist_order_payments p 
    LEFT JOIN olist_orders o ON p.order_id = o.order_id 
    WHERE o.order_id IS NULL
""", con=engine).iloc[0]['count']

# 3. Reviews without Orders
reviews_no_orders = pd.read_sql("""
    SELECT COUNT(*) AS count 
    FROM olist_order_reviews r 
    LEFT JOIN olist_orders o ON r.order_id = o.order_id 
    WHERE o.order_id IS NULL
""", con=engine).iloc[0]['count']

# 4. Order Items without Products
items_no_products = pd.read_sql("""
    SELECT COUNT(*) AS count 
    FROM olist_order_items i 
    LEFT JOIN olist_products p ON i.product_id = p.product_id 
    WHERE p.product_id IS NULL
""", con=engine).iloc[0]['count']

# 5. Products without Category Translation
products_no_translation = pd.read_sql("""
    SELECT COUNT(*) AS count 
    FROM olist_products p 
    LEFT JOIN product_category_translation t ON p.product_category_name = t.product_category_name 
    WHERE p.product_category_name IS NOT NULL AND t.product_category_name IS NULL
""", con=engine).iloc[0]['count']

# Summary Printout
print("🔗 RELATIONAL INTEGRITY REPORT:")
print(f"• Items without Orders: {items_no_orders}")
print(f"• Payments without Orders: {payments_no_orders}")
print(f"• Reviews without Orders: {reviews_no_orders}")
print(f"• Items without Products: {items_no_products}")
print(f"• Products missing English Translation: {products_no_translation}")

🔗 RELATIONAL INTEGRITY REPORT:
• Items without Orders: 0
• Payments without Orders: 0
• Reviews without Orders: 0
• Items without Products: 0
• Products missing English Translation: 13


## 3. Key Takeaways for Data Cleaning (Notebook 03)

1. **Relational Integrity:** Almost perfect core integrity across foreign keys (0 orphaned orders, items, or payments).
2. **Missing Category Translations:** Found **13** product categories that lack English translations. Need to manually map or label them as `unknown`/`other` during cleaning.
3. **Primary Key Health:** All primary keys verified as 100% unique across main entities (`order_id`, `customer_id`, `product_id`, etc.).
4. **Data Types:** Date/timestamp columns are currently stored as strings/objects and need explicit conversion to datetime objects in the cleaning phase.